In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import h5py
import os
from pathlib import Path

from enum import Enum
import re

from scipy.stats import skew, kurtosis
from scipy.fft import fft, fftfreq

In [ ]:
mat_file = r'E:\Thesis\thesis_code\data\rp\10Mbps\rp_ethernet_packets_1_2500.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_2501_5000.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_5001_7500.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_7501_10000.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_10001_12500.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_5cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_10cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_15cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_20cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_25cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_30cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_35cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_40cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_45cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_50cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_5cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_10cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_15cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_20cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_25cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_30cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_35cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_40cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_45cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_50cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\tapped\ethernet_packets_2500_0.5m.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\tapped\ethernet_packets_2500_1m.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\tapped\ethernet_packets_2500_1.5m.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\tapped\ethernet_packets_2500_2m.mat'


In [ ]:
with h5py.File(mat_file, 'r') as f:
    reference_packet = np.array(f['packets'])  # shape: (recordLength,)
    # Access metadata
    
    metadata = f['metadata']
    sample_rate = metadata['sample_rate'][0][0]
    trigger_level = metadata['trigger_level'][0][0]
    record_length = metadata['record_length'][0][0]
    num_frames = metadata['num_frames'][0][0]

print(f"Sample Rate: {sample_rate} Hz")
print(f"Trigger Level: {trigger_level} V")
print(f"Record Length: {record_length} samples")
print(f"Number of Frames: {num_frames}")

### Exract Signal Region

In [ ]:
def extract_signal_region(signal, threshold=0.1):
    """Trims the quiet parts of the signal before and after the packet."""
    active = np.abs(signal) > threshold
    indices = np.where(active)[0]
    if indices.size == 0:
        return np.array([])
    return signal[indices[0] : indices[-1] + 1]

### Ideal Packet Length

In [ ]:
sample_rate = 1.25e8  # 1.25GHz
samples_per_bit = int(sample_rate / 10e6)  # 10Mbps
samples_per_bit

In [ ]:
(sample_rate / 10e6)

In [ ]:
ideal_packet_length = (sample_rate / 10e6)*(8+148+4)*8
ideal_packet_length

### Sequence Padding 
- longer packets are discarded


In [ ]:
def length_standardization(extracted_signal, target_length=16000):
    """
    Standardizes signal to target_length. 
    Validates that the input is within a +/- 500 sample tolerance before padding.
    """
    current_len = len(extracted_signal)
    tolerance = 500 

    # 1. Check if it's too long
    if current_len > (target_length + tolerance):
        raise ValueError(f"Signal ({current_len}) exceeds target + tolerance.")

    # 2. Check if it's too short
    if current_len < (target_length - tolerance):
        raise ValueError(f"Signal ({current_len}) is below target - tolerance.")

    # 3. Standardize
    if current_len > target_length:
        # If within tolerance but slightly over, clip it
        return extracted_signal[:target_length]
    else:
        # If within tolerance but slightly under, pad it
        pad_size = target_length - current_len
        return np.pad(extracted_signal, (0, pad_size), mode='constant')

### Extract Time domain Features

In [ ]:
def extract_time_domain_features(signal):
    """Extract time-domain statistical features"""
    features = {}
    
    # Basic statistical moments
    features['std'] = np.std(signal)
    features['skewness'] = skew(signal)
    features['rms'] = np.sqrt(np.mean(signal**2))
    features['mean'] = np.mean(signal)
    features['kurtosis'] = kurtosis(signal)
    features['peak'] = np.max(np.abs(signal))
    
    # Shape factors
    features['shape_factor'] = features['rms'] / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) != 0 else 0
    features['impulse_factor'] = features['peak'] / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) != 0 else 0
    features['crest_factor'] = features['peak'] / features['rms'] if features['rms'] != 0 else 0
    features['clearance_factor'] = features['peak'] / (np.mean(np.sqrt(np.abs(signal)))**2) if np.mean(np.sqrt(np.abs(signal))) != 0 else 0
    
    return features

### Extract Frequency domain features

In [ ]:
def extract_frequency_domain_features(signal, fs):
    """Extract frequency-domain features"""
    # Compute FFT
    fft_signal = fft(signal)
    freqs = fftfreq(len(signal), 1/fs)
    
    # Get positive frequencies only
    positive_freq_idx = freqs > 0
    fft_positive = fft_signal[positive_freq_idx]
    freqs_positive = freqs[positive_freq_idx]
    
    # Find top 10 harmonic components
    magnitudes = np.abs(fft_positive)
    top_indices = np.argsort(magnitudes)[-10:]
    
    features = {}
    for i, idx in enumerate(top_indices):
        features[f'freq_mag_{i}'] = magnitudes[idx]
        features[f'freq_val_{i}'] = freqs_positive[idx]
    
    return features

### Label Extracton from File Name

In [ ]:
class ChannelCondition(Enum):
    NORMAL = 0
    AIR = 1
    WATER = 2
    TAPPED = 3

In [ ]:
def extract_label_from_filename(filepath):
    filename = filepath.lower()

    # TYPE LABEL
    if 'air' in filename:
        anomaly_type = ChannelCondition.AIR.value
    elif 'water' in filename:
        anomaly_type = ChannelCondition.WATER.value
    elif 'tapped' in filename:
        anomaly_type = ChannelCondition.TAPPED.value
    else:
        anomaly_type = ChannelCondition.NORMAL.value  # normal

    # LENGTH / DISTANCE
    
    match = re.search(r'_(\d+(?:\.\d+)?)(cm|m)\.mat$', filename)

    if match:
        value = float(match.group(1))
        unit = match.group(2)

        if unit == 'm':
            value *= 100  # convert to cm

        anomaly_length = value
    else:
        anomaly_length = 0

    return anomaly_type, anomaly_length

In [ ]:
# extract_label_from_filename(mat_file)

In [ ]:
from pathlib import Path

root_dir = Path(r"E:\Thesis\thesis_code\data\rp\10Mbps")

for mat_file in root_dir.rglob("*.mat"):
    print(mat_file)
    print(extract_label_from_filename(str(mat_file)))

In [ ]:
output_file = r'E:\Thesis\thesis_code\data\rp\10Mbps_features.h5'

In [ ]:

def extract_features_from_mat_file(root_dir, output_file):
    
    with h5py.File(output_file, 'x') as out_f:

        feature_dim = 10 + 20  # 10 time + (10 mag + 10 freq)

        features_dset = out_f.create_dataset(
            'features',
            shape=(0, feature_dim),
            maxshape=(None, feature_dim),
            dtype='float32',
            chunks=True
        )

        labels_type = out_f.create_dataset(
            'label_type',
            shape=(0,),
            maxshape=(None,),
            dtype='int32',
            chunks=True
        )

        labels_length = out_f.create_dataset(
            'label_length',
            shape=(0,),
            maxshape=(None,),
            dtype='float32',
            chunks=True
        )

        current_size = 0

        for mat_file in root_dir.rglob("*.mat"):
            print(f"\nProcessing: {mat_file}")

            anomaly_type, anomaly_length = extract_label_from_filename(str(mat_file))

            with h5py.File(mat_file, 'r') as f:
                packets = f['packets']
                metadata = f['metadata']

                fs = metadata['sample_rate'][0][0]
                num_frames = packets.shape[0]

                batch_size = 100
                for i in range(0, num_frames, batch_size):
                    

                    try:
                        batch = packets[i:i+batch_size]  # lazy load
                        features_batch = []
                        types_batch = []
                        lengths_batch = []
                        for j, signal in enumerate(batch):
                            try:
                                # --- preprocessing ---
                                signal = extract_signal_region(signal)


                                signal = length_standardization(signal)

                                # --- feature extraction ---
                                time_feat = extract_time_domain_features(signal)
                                freq_feat = extract_frequency_domain_features(signal, fs)

                                # Combine features
                                combined = list(time_feat.values()) + list(freq_feat.values())
                                combined = np.array(combined, dtype=np.float32)

                                # --- store ---
                                # features_dset.resize(current_size + 1, axis=0)
                                # labels_type.resize(current_size + 1, axis=0)
                                # labels_length.resize(current_size + 1, axis=0)

                                # features_dset[current_size] = combined
                                # labels_type[current_size] = anomaly_type
                                # labels_length[current_size] = anomaly_length

                                features_batch.append(combined)
                                types_batch.append(anomaly_type)
                                lengths_batch.append(anomaly_length)
                            except Exception as e:
                                print(f"Skipping signal in frame {i+j}: {e}")
                                continue

                        # --- ONLY write valid samples ---
                        valid_count = len(features_batch)


                        if valid_count == 0:
                            continue

                        new_size = current_size + valid_count

                        features_dset.resize(new_size, axis=0)
                        labels_type.resize(new_size, axis=0)
                        labels_length.resize(new_size, axis=0)

                        features_dset[current_size:new_size] = features_batch
                        labels_type[current_size:new_size] = types_batch
                        labels_length[current_size:new_size] = lengths_batch

                        current_size = new_size

                    except Exception as e:
                        print(f"Skipping frame {i}: {e}")
                        continue

                    if i % 100 == 0:
                        print(f"Processed {i}/{num_frames} | Added {valid_count} samples")
                    # break
            # break
    print("\n✅ Feature extraction completed!")

In [ ]:
# import cProfile
# cProfile.run('extract_features_from_mat_file(root_dir, output_file)')

In [ ]:
extract_features_from_mat_file(root_dir, output_file)